## Importing Packages

In [4]:
import sys
!{sys.executable} -m pip install xgboost mlflow dagshub imbalanced-learn

Defaulting to user installation because normal site-packages is not writeable
  Using cached dagshub-0.7.1-py3-none-any.whl.metadata (12 kB)
  Using cached dacite-1.6.0-py3-none-any.whl.metadata (14 kB)
  Using cached gql-4.0.0-py3-none-any.whl.metadata (10 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached treelib-1.8.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached pathvalidate-3.3.1-py3-none-any.whl.metadata (12 kB)
  Using cached boto3-1.43.66-py3-none-any.whl.metadata (6.6 kB)
  Using cached dagshub_annotation_converter-0.2.2-py3-none-any.whl.metadata (4.6 kB)
  Using cached botocore-1.43.66-py3-none-any.whl.metadata (5.6 kB)
  Using cached s3transfer-0.19.2-py3-none-any.whl.metadata (1.7 kB)
  Using cached marshmallow-3.26.2-py3-none-any.whl.metadata (7.3 kB)
  Using cached typing_inspect-0.9.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached backoff-2.2.1-py3-none-any.whl.metadata (14 kB)
Using cached dagshub-0.7.1-py3-none-any.whl (27

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aiobotocore 2.25.0 requires botocore<1.40.50,>=1.40.46, but you have botocore 1.43.66 which is incompatible.


In [5]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

import mlflow
import mlflow.sklearn
import mlflow.xgboost

import dagshub
import os

## DagsHub MLflow Setup

In [9]:
dagshub.init(
    repo_owner="mohnish1234-git",
    repo_name="dev_ops_056",
    mlflow=True
)

Initialized MLflow to track repo "mohnish1234-git/dev_ops_056"

Repository mohnish1234-git/dev_ops_056 initialized!

In [10]:
mlflow.set_experiment(
    "Boston Housing Regression PBLM 1"
)

2026/08/07 11:19:20 INFO mlflow.tracking.fluent: Experiment with name 'Boston Housing Regression PBLM 1' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/a5e41f61baf644e18d14388d5912171f', creation_time=1786081761730, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1786081761730, lifecycle_stage='active', name='Boston Housing Regression PBLM 1', tags={}, trace_location=None, workspace='default'>

## Data Loading and Processing

In [11]:
import pandas as pd

df = pd.read_csv("BostonHousing.csv")

X = df.drop("medv", axis=1)
y = df["medv"]

print(X.shape)
print(y.shape)

(506, 13)
(506,)


In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42
)

In [13]:
X_train_res = X_train.copy()
y_train_res = y_train.copy()

print(X_train_res.shape)
print(y_train_res.shape)

(354, 13)
(354,)


## Build Models

In [14]:
models = [
    (
        "Linear Regression",
        LinearRegression(),
        X_train,
        y_train
    ),
    (
        "Random Forest Regressor",
        RandomForestRegressor(
            n_estimators=100,
            max_depth=5,
            random_state=42
        ),
        X_train,
        y_train
    ),
    (
        "XGBoost Regressor",
        XGBRegressor(
            random_state=42
        ),
        X_train,
        y_train
    )
]

In [15]:
print(df.isnull().sum())

crim       0
zn         0
indus      0
chas       0
nox        0
rm         0
age        0
dis        0
rad        0
tax        0
ptratio    0
b          0
lstat      0
medv       0
dtype: int64


In [16]:
reports = []
trained_models = []

for model_name, model, X_tr, y_tr in models:
    model.fit(X_tr, y_tr)

    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    rmse = mse ** 0.5
    r2 = r2_score(y_test, predictions)

    report = {
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    }

    reports.append(report)
    trained_models.append(model)

    print("=" * 50)
    print(model_name)
    print("=" * 50)
    print(f"MAE  : {mae:.4f}")
    print(f"MSE  : {mse:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R2   : {r2:.4f}")

Linear Regression
MAE  : 3.1627
MSE  : 21.5174
RMSE : 4.6387
R2   : 0.7112
Random Forest Regressor
MAE  : 2.2751
MSE  : 11.1488
RMSE : 3.3390
R2   : 0.8504
XGBoost Regressor
MAE  : 2.0600
MSE  : 9.2880
RMSE : 3.0476
R2   : 0.8754


## Log All Experiments to DagsHub

In [17]:
for i, (model_name, model, _, _) in enumerate(models):
    report = reports[i]

    with mlflow.start_run(run_name=model_name):

        # Log model name
        mlflow.log_param(
            "Model",
            model_name
        )

        # Log hyperparameters
        mlflow.log_params(
            model.get_params()
        )

        # Log regression metrics
        mlflow.log_metric(
            "MAE",
            report["MAE"]
        )

        mlflow.log_metric(
            "MSE",
            report["MSE"]
        )

        mlflow.log_metric(
            "RMSE",
            report["RMSE"]
        )

        mlflow.log_metric(
            "R2",
            report["R2"]
        )

        # Log model
        if "XGBoost" in model_name:

            mlflow.xgboost.log_model(
                model,
                "model"
            )

        else:

            mlflow.sklearn.log_model(
                model,
                "model"
            )

print("Experiments logged!")

2026/08/07 11:19:33 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Linear Regression at: https://dagshub.com/mohnish1234-git/dev_ops_056.mlflow/#/experiments/1/runs/299d5c87f7ea4880a94f75afec2d7ddb
🧪 View experiment at: https://dagshub.com/mohnish1234-git/dev_ops_056.mlflow/#/experiments/1


2026/08/07 11:20:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest Regressor at: https://dagshub.com/mohnish1234-git/dev_ops_056.mlflow/#/experiments/1/runs/55922b0ef39c4afab94a9d01f78833b8
🧪 View experiment at: https://dagshub.com/mohnish1234-git/dev_ops_056.mlflow/#/experiments/1


2026/08/07 11:21:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBoost Regressor at: https://dagshub.com/mohnish1234-git/dev_ops_056.mlflow/#/experiments/1/runs/ca041b8a1f1748cea163eb0894bd4652
🧪 View experiment at: https://dagshub.com/mohnish1234-git/dev_ops_056.mlflow/#/experiments/1
Experiments logged!


## Best Model and Reg to DH

In [19]:
best_index = np.argmax(
    [
        r["R2"]
        for r in reports
    ]
)

best_model_name = models[best_index][0]

best_model = trained_models[best_index]

best_report = reports[best_index]

print("Best Model:", best_model_name)
print("R2 Score:", best_report["R2"])

Best Model: XGBoost Regressor
R2 Score: 0.8753514389204832


In [20]:
with mlflow.start_run(
    run_name=f"Champion_{best_model_name}"
) as run:

    mlflow.log_param(
        "Model",
        best_model_name
    )

    mlflow.log_metric(
        "MAE",
        best_report["MAE"]
    )

    mlflow.log_metric(
        "MSE",
        best_report["MSE"]
    )

    mlflow.log_metric(
        "RMSE",
        best_report["RMSE"]
    )

    mlflow.log_metric(
        "R2",
        best_report["R2"]
    )

    if "XGBoost" in best_model_name:

        mlflow.xgboost.log_model(
            best_model,
            "model",
            registered_model_name="Boston_Housing_Best_Model"
        )

    else:

        mlflow.sklearn.log_model(
            best_model,
            "model",
            registered_model_name="Boston_Housing_Best_Model"
        )

    run_id = run.info.run_id

print(run_id)

2026/08/07 11:22:09 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'Boston_Housing_Best_Model'.
2026/08/07 11:22:44 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Boston_Housing_Best_Model, version 1
Created version '1' of model 'Boston_Housing_Best_Model'.


🏃 View run Champion_XGBoost Regressor at: https://dagshub.com/mohnish1234-git/dev_ops_056.mlflow/#/experiments/1/runs/fca320bab56d44a780a2e6dc64c03de8
🧪 View experiment at: https://dagshub.com/mohnish1234-git/dev_ops_056.mlflow/#/experiments/1
fca320bab56d44a780a2e6dc64c03de8
